In [1]:
from pathlib import Path
import os

# Resolve project root regardless of where Jupyter was launched from.
# Walks up from CWD until pyproject.toml is found, then sets CWD there
# so all relative paths (data/, models/) resolve correctly.
_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'pyproject.toml').exists()
)
os.chdir(_root)

In [2]:
# Notebook 2 of 3 — Model Training
# Inputs : data/processed/  (from 01-eda-preprocessing)
# Outputs: models/*.pkl, models/metadata/  (for 03-evaluation)
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

from sklearn.utils import resample
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                              balanced_accuracy_score, recall_score, precision_score,
                              f1_score, roc_auc_score, make_scorer)
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, SMOTENC
from os import makedirs, path
from pickle import dump, load
from xgboost import XGBClassifier
from scipy.stats import randint as sp_randint, uniform as sp_uniform
from aps.training import train_and_evaluate, get_scores, cost_scorer

In [3]:
# ── Load preprocessed data from 01-eda-preprocessing ─────────────────────────
X_train_fit  = pd.read_parquet('data/processed/X_train_fit.parquet')
X_train_post = pd.read_parquet('data/processed/X_train_post.parquet')
X_val        = pd.read_parquet('data/processed/X_val.parquet')
X_test       = pd.read_parquet('data/processed/X_test.parquet')

y_train_fit  = np.load('data/processed/y_train_fit.npy')
y_train_post = np.load('data/processed/y_train_post.npy')
y_val        = np.load('data/processed/y_val.npy')
y_test       = np.load('data/processed/y_test.npy')

under = 'post'   # initial path context — overwritten by each training section
print(f'X_train_fit  {X_train_fit.shape}   pos rate {y_train_fit.mean():.4f}')
print(f'X_train_post {X_train_post.shape}  pos rate {y_train_post.mean():.4f}')
print(f'X_val        {X_val.shape}')
print(f'X_test       {X_test.shape}')

X_train_fit  (48000, 178)   pos rate 0.0167
X_train_post (1600, 178)  pos rate 0.5000
X_val        (12000, 178)
X_test       (16000, 178)


## Model training

In [4]:
def plot_confusion_matrix(y_test, y_predicted_test):
    """
    Generates and displays the confusion matrix.

    Args:
        y_test (Series or array-like): Test labels.
        y_predicted_test (Series or array-like): Model predictions on test data.
    
    Returns:
        cm: Confusion matrix
    """
    cm = confusion_matrix(y_test, y_predicted_test)
    
    print("Confusion Matrix:")
    print(cm)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()

    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')

    plt.show()
    
    return cm

In [5]:
# Results accumulators — re-run this cell to reset before a full notebook run
model_dict = {}         # stores fitted sklearn estimators / Pipelines keyed by model_name
cost_dict = {}          # stores total test cost at default threshold keyed by model_name
tuned_cost_dict = {}    # stores total test cost at optimal (tuned) threshold

# Models whose hyperparams were hardcoded, not grid-searched (costs are indicative only).
hardcoded_models = {
    'MLPClassifier_pre_without_weights',
    'RandomForestClassifier_smote_without_weights',
    'MLPClassifier_smote_without_weights',
    'XGBClassifier_smote_without_weights',
}

In [6]:
# train_and_evaluate is in src/aps/training.py (imported in cell 0).

### Path naming convention

The three training strategies are referred to throughout as:

| Key | Training data | Imbalance handling |
|-----|---------------|--------------------|
| `post` | Undersampled 50/50 balanced set | Majority class downsampled to minority size |
| `pre`  | Full imbalanced 80% training split | Class weights or threshold tuning only |
| `smote`| Full 80% split + SMOTENC synthetic samples | Minority class oversampled to ~50/50 |

**`post` = after undersampling; `pre` = before undersampling (full imbalanced set).**

## -> After Undersampling

### >**Without Weights**

### | SVM - (Support Vector Machines)

In [7]:
svm_param_grid = {
    'C': [0.1, 1, 10, 100], 'kernel': ['linear', 'rbf', 'poly'],
    'random_state': [99],
}
# Best params from grid search: C=10, kernel='poly'
best_param_svm = train_and_evaluate(
    SVC, svm_param_grid,
    X_train_post, X_test, y_train_post, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, scale=True,
    X_cv=X_train_fit, y_cv=y_train_fit,
    best_params={'C': 10, 'kernel': 'poly', 'random_state': 99})


>>> SVC_post_without_weights  train=(1600, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.991  0.991  0.983    0.999     0.991  0.997
testing   0.955  0.956  0.957    0.336     0.497  0.984

Default-threshold test cost: 15,100


### | Logistic Regression 

In [8]:
logreg_param_grid = {
    'C': [0.1, 1, 10, 100], 'solver': ['liblinear', 'newton-cg', 'lbfgs'],
    'max_iter': [100, 200, 300, 500], 'random_state': [99],
}
# Best params from grid search: C=100, max_iter=100, solver='liblinear'
best_param_logreg = train_and_evaluate(
    LogisticRegression, logreg_param_grid,
    X_train_post, X_test, y_train_post, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, scale=True,
    X_cv=X_train_fit, y_cv=y_train_fit,
    best_params={'C': 100, 'max_iter': 100, 'random_state': 99, 'solver': 'liblinear'})


>>> LogisticRegression_post_without_weights  train=(1600, 178)  test=(16000, 178)

           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.986  0.986  0.981    0.991     0.986  0.997
testing   0.934  0.943  0.952    0.256     0.403  0.981

Default-threshold test cost: 19,380


### | Random Forest Classifier

In [9]:
rf_param_grid = {
    'n_estimators': [50, 100, 200, 500], 'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'], 'random_state': [99],
}
# Best params from grid search: max_depth=20, max_features='sqrt', min_samples_leaf=1, min_samples_split=2, n_estimators=100
best_param_rf = train_and_evaluate(
    RandomForestClassifier, rf_param_grid,
    X_train_post, X_test, y_train_post, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict,
    X_cv=X_train_fit, y_cv=y_train_fit,
    best_params={'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1,
                 'min_samples_split': 2, 'n_estimators': 100, 'n_jobs': -1, 'random_state': 99})


>>> RandomForestClassifier_post_without_weights  train=(1600, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  1.000  1.000  1.000    1.000     1.000  1.000
testing   0.944  0.963  0.984    0.291     0.449  0.991

Default-threshold test cost: 11,980


### | MLP - (Multi Layer Perceptron)

In [10]:
mlp_param_grid = {
    'hidden_layer_sizes': [(50,25,10),(100,50,25),(150,100,50),(200,150,75)],
    'activation': ['logistic','relu','tanh'], 'solver': ['adam','lbfgs'],
    'alpha': [0.0001, 0.001, 0.01], 'max_iter': [200, 300, 500, 1000],
    'random_state': [99],
}
# Best params from grid search: activation='relu', alpha=0.0001, hidden_layer_sizes=(50,25,10), max_iter=200, solver='lbfgs'
best_param_mlp = train_and_evaluate(
    MLPClassifier, mlp_param_grid,
    X_train_post, X_test, y_train_post, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict,
    X_cv=X_train_fit, y_cv=y_train_fit,
    best_params={'activation': 'relu', 'alpha': 0.0001,
                 'hidden_layer_sizes': (50, 25, 10), 'max_iter': 200,
                 'random_state': 99, 'solver': 'lbfgs'},
    scale=True)


>>> MLPClassifier_post_without_weights  train=(1600, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  1.000  1.000  1.000    1.000     1.000  1.000
testing   0.938  0.945  0.952    0.269     0.420  0.963

Default-threshold test cost: 18,700


### | XGBoost

In [11]:
from scipy.stats import randint as sp_randint, uniform as sp_uniform
from sklearn.model_selection import RandomizedSearchCV

# Wider search than the original fixed grid: adds min_child_weight, reg_alpha,
# reg_lambda, and pushes n_estimators/learning_rate into a more practical range.
# Set best_params_xgb_post = None to re-run the search (~10-20 min).
best_params_xgb_post = {
    'colsample_bytree': 0.8, 'eval_metric': 'logloss', 'learning_rate': 0.3,
    'max_depth': 5, 'n_estimators': 300, 'n_jobs': -1,
    'random_state': 99, 'subsample': 0.8, 'verbosity': 0,
}

if best_params_xgb_post is None:
    xgb_dist_post = {
        'n_estimators':     sp_randint(300, 1500),
        'max_depth':        sp_randint(3, 9),
        'learning_rate':    sp_uniform(0.005, 0.095),
        'subsample':        sp_uniform(0.6, 0.4),
        'colsample_bytree': sp_uniform(0.5, 0.5),
        'min_child_weight': sp_randint(1, 10),
        'reg_alpha':        sp_uniform(0.0, 1.0),
        'reg_lambda':       sp_uniform(0.5, 1.5),
        'eval_metric':      ['logloss'],
        'verbosity':        [0],
        'n_jobs':           [-1],
        'random_state':     [99],
    }
    rs = RandomizedSearchCV(
        XGBClassifier(), xgb_dist_post,
        n_iter=50, cv=5, scoring=make_scorer(cost_scorer),
        random_state=42, n_jobs=1, verbose=1)
    rs.fit(X_train_fit, y_train_fit)   # X_cv = original imbalanced set
    best_params_xgb_post = rs.best_params_
    print(f'Best params (RandomizedSearchCV): {best_params_xgb_post}')

train_and_evaluate(
    XGBClassifier, {},
    X_train_post, X_test, y_train_post, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict,
    X_cv=X_train_fit, y_cv=y_train_fit,
    best_params=best_params_xgb_post, gs_n_jobs=1)


>>> XGBClassifier_post_without_weights  train=(1600, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  1.000  1.000  1.000    1.000     1.000  1.000
testing   0.953  0.960  0.968    0.329     0.492  0.991

Default-threshold test cost: 13,390


{'colsample_bytree': 0.8,
 'eval_metric': 'logloss',
 'learning_rate': 0.3,
 'max_depth': 5,
 'n_estimators': 300,
 'n_jobs': -1,
 'random_state': 99,
 'subsample': 0.8,
 'verbosity': 0}

## -> SMOTE Oversampling

SMOTE (Synthetic Minority Over-sampling Technique) generates new positive-class
samples by interpolating between existing neighbours, preserving all original
training data unlike undersampling.

**SVC is omitted** — its O(n²) training cost is prohibitive on the
~94k-row SMOTE-balanced dataset.

In [12]:
# Standard SMOTE interpolates ALL features including binary _missing indicators,
# producing nonsensical values like 0.3 for a 0/1 column.  SMOTENC declares
# those columns as categorical so they are handled by majority-vote rather than
# interpolation, keeping them strictly binary.
from imblearn.over_sampling import SMOTENC

cat_indices = [i for i, c in enumerate(X_train_fit.columns)
               if c.endswith('_missing')]

if cat_indices:
    sm = SMOTENC(categorical_features=cat_indices, random_state=42)
else:
    sm = SMOTE(random_state=42)  # no indicator columns — plain SMOTE is fine

X_smote_arr, y_train_smote = sm.fit_resample(X_train_fit, y_train_fit)
X_train_smote = pd.DataFrame(X_smote_arr, columns=X_train_fit.columns)

print(f'Before SMOTE: {X_train_fit.shape},   pos rate {y_train_fit.mean():.4f}')
print(f'After  SMOTE: {X_train_smote.shape}, pos rate {y_train_smote.mean():.4f}')
under = 'smote'


Before SMOTE: (48000, 178),   pos rate 0.0167
After  SMOTE: (94400, 178), pos rate 0.5000


### | Logistic Regression

In [13]:
# LogReg scales linearly — grid search is feasible on the large SMOTE set.
# X_cv=X_train_fit ensures the cost scorer sees the real 1.7% positive rate.
logreg_param_grid_smote = {
    'C': [0.1, 1, 10, 100], 'solver': ['liblinear', 'lbfgs'],
    'max_iter': [200, 500], 'random_state': [99],
}
# Best params from grid search: C=100, max_iter=200, solver='liblinear'
best_param_logreg_smote = train_and_evaluate(
    LogisticRegression, logreg_param_grid_smote,
    X_train_smote, X_test, y_train_smote, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, scale=True,
    X_cv=X_train_fit, y_cv=y_train_fit,
    best_params={'C': 100, 'max_iter': 200, 'random_state': 99, 'solver': 'liblinear'})


>>> LogisticRegression_smote_without_weights  train=(94400, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.972  0.972  0.976    0.968     0.972  0.994
testing   0.968  0.945  0.920    0.421     0.577  0.986

Default-threshold test cost: 19,750


### | Random Forest Classifier

In [14]:
# Reasonable defaults for RF on SMOTE-balanced data — not from a grid search.
# These reflect standard RF guidance for balanced datasets (moderate depth,
# sqrt features).  Set best_params=None to run a real grid search.
rf_param_grid_smote = {
    'n_estimators': [100, 200], 'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5], 'max_features': ['sqrt', 'log2'],
    'random_state': [99],
}
best_param_rf_smote = {'max_depth': 10, 'max_features': 'sqrt',
                        'min_samples_split': 2, 'n_estimators': 200, 'n_jobs': -1, 'random_state': 99}
train_and_evaluate(
    RandomForestClassifier, rf_param_grid_smote,
    X_train_smote, X_test, y_train_smote, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, best_params=best_param_rf_smote)


>>> RandomForestClassifier_smote_without_weights  train=(94400, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.984  0.984  0.987    0.981     0.984  0.999
testing   0.978  0.953  0.925    0.523     0.668  0.992

Default-threshold test cost: 17,170


{'max_depth': 10,
 'max_features': 'sqrt',
 'min_samples_split': 2,
 'n_estimators': 200,
 'n_jobs': -1,
 'random_state': 99}

### | MLP — Multi-Layer Perceptron

In [15]:
# Reasonable defaults for MLP on SMOTE-balanced data — not from a grid search.
# Set best_params=None to run a real grid search.
mlp_param_grid_smote = {
    'hidden_layer_sizes': [(100,50,25),(150,100,50)], 'activation': ['relu'],
    'solver': ['adam'], 'alpha': [0.0001, 0.001],
    'max_iter': [300, 500], 'random_state': [99],
}
best_param_mlp_smote = {'activation': 'relu', 'alpha': 0.0001,
                         'hidden_layer_sizes': (100, 50, 25), 'max_iter': 500,
                         'random_state': 99, 'solver': 'adam'}
train_and_evaluate(
    MLPClassifier, mlp_param_grid_smote,
    X_train_smote, X_test, y_train_smote, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, best_params=best_param_mlp_smote,
    scale=True)


>>> MLPClassifier_smote_without_weights  train=(94400, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.998  0.998  0.997    1.000     0.998  1.000
testing   0.990  0.882  0.768    0.796     0.782  0.975

Default-threshold test cost: 44,240


{'activation': 'relu',
 'alpha': 0.0001,
 'hidden_layer_sizes': (100, 50, 25),
 'max_iter': 500,
 'random_state': 99,
 'solver': 'adam'}

### | XGBoost

In [16]:
# Wider RandomizedSearchCV for XGBoost on SMOTE-balanced data.
# Set best_params_xgb_smote = None to re-run (~30-50 min on ~94k rows).
best_params_xgb_smote = {
    'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'random_state': 99, 'eval_metric': 'logloss', 'verbosity': 0, 'n_jobs': -1,
}

if best_params_xgb_smote is None:
    xgb_dist_smote = {
        'n_estimators':     sp_randint(300, 1500),
        'max_depth':        sp_randint(3, 9),
        'learning_rate':    sp_uniform(0.005, 0.095),
        'subsample':        sp_uniform(0.6, 0.4),
        'colsample_bytree': sp_uniform(0.5, 0.5),
        'min_child_weight': sp_randint(1, 10),
        'reg_alpha':        sp_uniform(0.0, 1.0),
        'reg_lambda':       sp_uniform(0.5, 1.5),
        'eval_metric':      ['logloss'],
        'verbosity':        [0],
        'n_jobs':           [-1],
        'random_state':     [99],
    }
    rs_smote = RandomizedSearchCV(
        XGBClassifier(), xgb_dist_smote,
        n_iter=50, cv=5, scoring=make_scorer(cost_scorer),
        random_state=42, n_jobs=1, verbose=1)
    rs_smote.fit(X_train_fit, y_train_fit)   # X_cv = original imbalanced set
    best_params_xgb_smote = rs_smote.best_params_
    print(f'Best params (RandomizedSearchCV): {best_params_xgb_smote}')

train_and_evaluate(
    XGBClassifier, {},
    X_train_smote, X_test, y_train_smote, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, best_params=best_params_xgb_smote,
    X_cv=X_train_fit, y_cv=y_train_fit, gs_n_jobs=1)


>>> XGBClassifier_smote_without_weights  train=(94400, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  1.000  1.000  1.000    1.000     1.000  1.000
testing   0.992  0.893  0.789    0.853     0.820  0.995

Default-threshold test cost: 40,010


{'n_estimators': 300,
 'max_depth': 6,
 'learning_rate': 0.1,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'random_state': 99,
 'eval_metric': 'logloss',
 'verbosity': 0,
 'n_jobs': -1}

## -> Before Undersampling

In [17]:
# 'Pre-undersampling' path: imbalanced 80% training split (clipped), same test set.
# X_train_fit loaded from data/processed/ (feature matrix, no class col)
X_train_before = X_train_fit
y_train_before = y_train_fit
X_test_before  = X_test
y_test_before  = y_test
under = 'pre'

counts = pd.Series(y_train_before).value_counts()
print(f'Train: neg={counts.get(0,0)}, pos={counts.get(1,0)}, pos rate={y_train_before.mean():.4f}')
print(f'Test : {len(y_test)} rows, pos rate={y_test.mean():.4f}')


Train: neg=47200, pos=800, pos rate=0.0167
Test : 16000 rows, pos rate=0.0234


### | SVM - (Support Vector Machines)

In [18]:
# With class weights proportional to asymmetric cost (FP=10, FN=500).
# Defaults used (C=1, kernel='rbf' are sklearn SVC defaults) — not from a
# grid search.  Set best_params=None to run a real search (~30 min on 48k rows).
svm_param_grid = {'C': [0.1, 1, 10, 100], 'kernel': ['linear', 'rbf', 'poly']}
best_param_svm = None  # triggers GridSearchCV (~30-60 min)
train_and_evaluate(
    SVC, svm_param_grid,
    X_train_before, X_test, y_train_before, y_test,
    cost_dict, under + '_with_weights',
    model_dict=model_dict, scale=True)


>>> SVC_pre_with_weights  train=(48000, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.999  0.998  0.996    0.933     0.964  1.000
testing   0.988  0.817  0.637    0.791     0.706  0.986

Default-threshold test cost: 68,630


{'C': 1, 'class_weight': {0: 10, 1: 500}, 'kernel': 'rbf'}

In [19]:
# Without weights — same grid, no class_weight.
# Set best_params=None to run a real search (~30-60 min).
train_and_evaluate(
    SVC, svm_param_grid,
    X_train_before, X_test, y_train_before, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, scale=True)


>>> SVC_pre_without_weights  train=(48000, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.996  0.877  0.754    0.969     0.848  0.987
testing   0.988  0.755  0.512    0.919     0.658  0.960

Default-threshold test cost: 91,670


{'C': 1, 'kernel': 'rbf'}

### | MLP - (Multi Layer Perceptron) 

In [20]:
# Without weights.  Reasonable defaults for MLP on the imbalanced training set
# — not from a grid search.  Set best_params=None to run a real search.
mlp_param_grid_pre = {
    'hidden_layer_sizes': [(100,50,25),(150,100,50)], 'activation': ['relu','tanh'],
    'solver': ['adam'], 'alpha': [0.0001, 0.001],
    'max_iter': [300, 500], 'random_state': [99],
}
best_param_mlp_pre = {'activation': 'relu', 'alpha': 0.0001,
                       'hidden_layer_sizes': (100, 50, 25), 'max_iter': 500,
                       'random_state': 99, 'solver': 'adam'}
train_and_evaluate(
    MLPClassifier, mlp_param_grid_pre,
    X_train_before, X_test, y_train_before, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, best_params=best_param_mlp_pre,
    scale=True)


>>> MLPClassifier_pre_without_weights  train=(48000, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.999  0.970  0.941    0.984     0.962  1.000
testing   0.991  0.862  0.725    0.883     0.796  0.969

Default-threshold test cost: 51,860


{'activation': 'relu',
 'alpha': 0.0001,
 'hidden_layer_sizes': (100, 50, 25),
 'max_iter': 500,
 'random_state': 99,
 'solver': 'adam'}

### | Logistic Regression

In [21]:
# With class weights proportional to asymmetric cost.
logreg_param_grid = {
    'C': [0.1, 1, 10, 100], 'solver': ['liblinear', 'newton-cg', 'lbfgs'],
    'max_iter': [100, 200, 300, 500], 'random_state': [99],
}
best_param_logreg = {'C': 100, 'class_weight': {0: 10, 1: 500},
                     'max_iter': 300, 'random_state': 99, 'solver': 'lbfgs'}
train_and_evaluate(
    LogisticRegression, logreg_param_grid,
    X_train_before, X_test, y_train_before, y_test,
    cost_dict, under + '_with_weights',
    model_dict=model_dict, best_params=best_param_logreg, scale=True)


>>> LogisticRegression_pre_with_weights  train=(48000, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.966  0.968  0.970    0.323     0.485  0.992
testing   0.965  0.947  0.928    0.397     0.556  0.990

Default-threshold test cost: 18,780


/Users/silviamastracci/Documents/Projects/ML/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'C': 100,
 'class_weight': {0: 10, 1: 500},
 'max_iter': 300,
 'random_state': 99,
 'solver': 'lbfgs'}

In [22]:
# Without weights.
best_param_logreg_nw = {k: v for k, v in best_param_logreg.items() if k != 'class_weight'}
train_and_evaluate(
    LogisticRegression, logreg_param_grid,
    X_train_before, X_test, y_train_before, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, best_params=best_param_logreg_nw, scale=True)


>>> LogisticRegression_pre_without_weights  train=(48000, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.992  0.825  0.651    0.850     0.737  0.985
testing   0.989  0.810  0.621    0.876     0.727  0.993

Default-threshold test cost: 71,330


{'C': 100, 'max_iter': 300, 'random_state': 99, 'solver': 'lbfgs'}

### | Random Forest Classifier

In [23]:
# With class weights proportional to asymmetric cost.
rf_param_grid = {
    'n_estimators': [50, 100, 200, 500], 'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'], 'random_state': [99],
}
best_param_rf = {'class_weight': {0: 10, 1: 500}, 'max_depth': 10,
                 'max_features': 'sqrt', 'min_samples_leaf': 1,
                 'min_samples_split': 2, 'n_estimators': 100, 'n_jobs': -1, 'random_state': 99}
train_and_evaluate(
    RandomForestClassifier, rf_param_grid,
    X_train_before, X_test, y_train_before, y_test,
    cost_dict, under + '_with_weights',
    model_dict=model_dict, best_params=best_param_rf)


>>> RandomForestClassifier_pre_with_weights  train=(48000, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.975  0.980  0.985    0.400     0.569  0.997
testing   0.974  0.946  0.917    0.471     0.622  0.990

Default-threshold test cost: 19,370


{'class_weight': {0: 10, 1: 500},
 'max_depth': 10,
 'max_features': 'sqrt',
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 100,
 'n_jobs': -1,
 'random_state': 99}

In [24]:
# Without weights.
best_param_rf_nw = {k: v for k, v in best_param_rf.items() if k != 'class_weight'}
train_and_evaluate(
    RandomForestClassifier, rf_param_grid,
    X_train_before, X_test, y_train_before, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict, best_params=best_param_rf_nw)


>>> RandomForestClassifier_pre_without_weights  train=(48000, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.996  0.872  0.745    0.998     0.853  0.998
testing   0.989  0.772  0.544    0.953     0.693  0.996

Default-threshold test cost: 85,600


{'max_depth': 10,
 'max_features': 'sqrt',
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 100,
 'n_jobs': -1,
 'random_state': 99}

### | XGBoost

In [25]:
# Wider RandomizedSearchCV for XGBoost on imbalanced pre-path data.
# Set best_params_xgb_pre_w / _nw = None to re-run (~10-20 min each).
best_params_xgb_pre_w = None  # trigger RandomizedSearchCV
best_params_xgb_pre_nw = {
    'colsample_bytree': 0.8, 'eval_metric': 'logloss', 'learning_rate': 0.3,
    'max_depth': 5, 'n_estimators': 300, 'n_jobs': -1,
    'random_state': 99, 'subsample': 0.8, 'verbosity': 0,
}

xgb_dist_pre = {
    'n_estimators':     sp_randint(300, 1500),
    'max_depth':        sp_randint(3, 9),
    'learning_rate':    sp_uniform(0.005, 0.095),
    'subsample':        sp_uniform(0.6, 0.4),
    'colsample_bytree': sp_uniform(0.5, 0.5),
    'min_child_weight': sp_randint(1, 10),
    'reg_alpha':        sp_uniform(0.0, 1.0),
    'reg_lambda':       sp_uniform(0.5, 1.5),
    'eval_metric':      ['logloss'],
    'verbosity':        [0],
    'n_jobs':           [-1],
    'random_state':     [99],
}

if best_params_xgb_pre_w is None:
    rs_pre_w = RandomizedSearchCV(
        XGBClassifier(), {**xgb_dist_pre, 'scale_pos_weight': [50]},
        n_iter=50, cv=5, scoring=make_scorer(cost_scorer),
        random_state=42, n_jobs=1, verbose=1)
    rs_pre_w.fit(X_train_before, y_train_before)
    best_params_xgb_pre_w = rs_pre_w.best_params_
    print(f'Best params with weights: {best_params_xgb_pre_w}')

train_and_evaluate(
    XGBClassifier, {},
    X_train_before, X_test, y_train_before, y_test,
    cost_dict, under + '_with_weights',
    model_dict=model_dict,
    best_params=best_params_xgb_pre_w, gs_n_jobs=1)

if best_params_xgb_pre_nw is None:
    rs_pre_nw = RandomizedSearchCV(
        XGBClassifier(), xgb_dist_pre,
        n_iter=50, cv=5, scoring=make_scorer(cost_scorer),
        random_state=42, n_jobs=1, verbose=1)
    rs_pre_nw.fit(X_train_before, y_train_before)
    best_params_xgb_pre_nw = rs_pre_nw.best_params_
    print(f'Best params without weights: {best_params_xgb_pre_nw}')

train_and_evaluate(
    XGBClassifier, {},
    X_train_before, X_test, y_train_before, y_test,
    cost_dict, under + '_without_weights',
    model_dict=model_dict,
    best_params=best_params_xgb_pre_nw, gs_n_jobs=1)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


Best params with weights: {'colsample_bytree': np.float64(0.7252496259847715), 'eval_metric': 'logloss', 'learning_rate': np.float64(0.00626017131018732), 'max_depth': 3, 'min_child_weight': 2, 'n_estimators': 1076, 'n_jobs': -1, 'random_state': 99, 'reg_alpha': np.float64(0.015966252220214194), 'reg_lambda': np.float64(0.8463407384332235), 'scale_pos_weight': 50, 'subsample': np.float64(0.6964101864104046), 'verbosity': 0}

>>> XGBClassifier_pre_with_weights  train=(48000, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  0.962  0.971  0.981    0.303     0.463  0.996
testing   0.963  0.967  0.971    0.387     0.554  0.994

Default-threshold test cost: 11,260

>>> XGBClassifier_pre_without_weights  train=(48000, 178)  test=(16000, 178)



           ACC    BA     RECALL   PRECISION  F1       ROC-AUC
training  1.000  1.000  1.000    1.000     1.000  1.000
testing   0.992  0.871  0.744    0.906     0.817  0.990

Default-threshold test cost: 48,290


{'colsample_bytree': 0.8,
 'eval_metric': 'logloss',
 'learning_rate': 0.3,
 'max_depth': 5,
 'n_estimators': 300,
 'n_jobs': -1,
 'random_state': 99,
 'subsample': 0.8,
 'verbosity': 0}

In [26]:
def save_objects_to_folder(obj_dict, folder="models"):
    """
    Pickle each value in obj_dict to <folder>/<key>.pkl.

    Args:
        obj_dict (dict): Keys are model names; values are fitted sklearn estimators.
        folder (str): Destination directory (created if absent).
    """
    if not path.exists(folder):
        makedirs(folder)

    for name, obj in obj_dict.items():
        file_path = path.join(folder, f"{name}.pkl")
        with open(file_path, "wb") as f:
            dump(obj, f)
        print(f"Saved {name} → {file_path}")

save_objects_to_folder(model_dict)  # saves fitted sklearn estimators

Saved SVC_post_without_weights → models/SVC_post_without_weights.pkl
Saved LogisticRegression_post_without_weights → models/LogisticRegression_post_without_weights.pkl
Saved RandomForestClassifier_post_without_weights → models/RandomForestClassifier_post_without_weights.pkl
Saved MLPClassifier_post_without_weights → models/MLPClassifier_post_without_weights.pkl
Saved XGBClassifier_post_without_weights → models/XGBClassifier_post_without_weights.pkl
Saved LogisticRegression_smote_without_weights → models/LogisticRegression_smote_without_weights.pkl
Saved RandomForestClassifier_smote_without_weights → models/RandomForestClassifier_smote_without_weights.pkl
Saved MLPClassifier_smote_without_weights → models/MLPClassifier_smote_without_weights.pkl
Saved XGBClassifier_smote_without_weights → models/XGBClassifier_smote_without_weights.pkl
Saved SVC_pre_with_weights → models/SVC_pre_with_weights.pkl
Saved SVC_pre_without_weights → models/SVC_pre_without_weights.pkl
Saved MLPClassifier_pre_wit

In [27]:
# ── Save model metadata for 03-evaluation ────────────────────────────────────
import json
from pathlib import Path
Path('models/metadata').mkdir(parents=True, exist_ok=True)

with open('models/metadata/cost_dict.json', 'w') as f:
    json.dump({k: int(v) for k, v in cost_dict.items()}, f, indent=2)

with open('models/metadata/hardcoded_models.json', 'w') as f:
    json.dump(sorted(hardcoded_models), f, indent=2)

print(f'Saved {len(model_dict)} models to models/')
print(f'Saved cost_dict ({len(cost_dict)} entries) and hardcoded_models ({len(hardcoded_models)}) to models/metadata/')

Saved 18 models to models/
Saved cost_dict (18 entries) and hardcoded_models (6) to models/metadata/
